# NS-Spec v0.3 — Self-Verifying Notebook Specifications

**Status:** Approved design — supersedes v0.2  
**Primary solver:** Z3, reached *exclusively* through the existing SPUR `solve` MCP  
**Authoring and execution surface:** `.spec.ipynb` with native `ns-mermaid` cells  
**Portable artifact:** generated `.spec.md` snapshot (non-authoritative)  
**Review path:** retained as `ns-spec-v0.2-design.ipynb`; rename to v0.3 after spec review

> In v0.3, the notebook is the specification. Every formal unit is one
> executable `ns-mermaid` cell: its source visually declares topology,
> constraints, and named `@verify` points; running that same cell parses the
> declaration, lowers it to Canonical IR, verifies it through `solve`, and
> attaches proofs or counterexamples as cell outputs. No Python Z3 binding and
> no separately authored verification cell exist.

### Contents
1. Executive summary — notebook as spec
2. Same-cell architecture
3. Core principles
4. Per-cell verification decision flow and determinism
5. Verification menu
6. Conformance-test generation
7. `solve` integration and cell ports
8. Lifecycle and statuses
9. NS-Mermaid grammar and milestones
10. Solver-substrate evidence and verifier caveats
11. Normative `ns-mermaid` cell contract, procedure, and adoption conditions
12. Brain/worker self-correction, bootstrap trust, and immutable-spec repair
13. Compiler-backed agent authoring for the customized NS-Mermaid dialect
14. Executable complex-profile confirmation and observed runtime gaps
15. Contract-driven implementation and conformance testing

## 1. Executive summary

NS-Spec v0.3 makes a notebook cell the smallest authoritative specification
unit. An author writes annotated Mermaid; the same cell run renders the diagram,
parses its formal annotations, constructs Canonical IR, generates named proof
obligations, calls the existing `solve` MCP, and stores results in that cell's
outputs. A human approves the normalized meaning and fresh proof report before
implementation proceeds.

| Aspect | v0.2 | v0.3 |
|---|---|---|
| Authority | Markdown + `spec-yaml` | annotated Mermaid source in an `ns-mermaid` cell |
| Visualization | separate rendering | the declaration itself is the diagram |
| Verification | separately orchestrated audit | same-cell execution and outputs |
| Formal engine | `solve_constraints` / `solve_smt` | unchanged; still exclusive |
| Generated state | IR, obligations, reports | read-only cell outputs and typed ports |
| Portable Markdown | authorable profile | generated, non-authoritative snapshot |
| Acceptance | bounded conformance vs proof | unchanged, but scoped and hashed per cell |

### Why the same-cell model

A declaration and its verdict cannot drift into separate authored artifacts. The
cell source is pure and reviewable; execution never rewrites it. The cell runner
emits rendered Mermaid, normalized IR, individual obligation results, and an
aggregate proof report. Downstream report and conformance cells consume those
outputs through the reactive DAG.

## 2. Same-cell architecture

> **Executable diagram:** the following native `ns_mermaid` cell is the
authoritative architecture gate. It preserves the source-to-ports pipeline and
formally proves the ready/blocked partition.

This is literally one notebook cell: one source record, one execution, and one
output bundle. The `solve` call happens inside the `ns-mermaid` cell runner. It
is not a Python kernel call and not a separately authored verification cell.
The cell may emit outputs and ports, but MUST NOT rewrite its own source, consume
its own emitted ports, or mutate any cell it depends on; those rules keep the
reactive graph acyclic.

The runner is kernel-independent and constrained. Its source MIME is
`application/vnd.spur.ns-mermaid+text`; its structured result MIME is
`application/vnd.spur.ns-proof+json`. The ordinary Mermaid renderer consumes the
same source and displays annotations as label text; the NS parser reads those
annotations from source, never from the rendered DOM.

Each native cell publishes five producer names derived from its stable cell
identity: `spec_ir`, `proof_report`, `verified`, `conformance_vectors`, and
`diagnostics`.
Notebook-global aliases are forbidden because two formal cells must coexist
without producer collisions.

In [ ]:
flowchart TD
    CTX["`@spec NS-SPEC-SAME-CELL-ARCHITECTURE
@type ArchitectureDecision = enum[ready, blocked]
@input source_present: Bool
@input parser_pass: Bool
@input obligations_complete: Bool
@input solver_terminal: Bool
@input ports_published: Bool
@output decision: ArchitectureDecision`"]

    subgraph CELL[one executable ns-mermaid cell]
        SRC[annotated Mermaid source]
        PARSE[strict parser and type checker]
        IR[Canonical IR plus source map]
        PLAN[named proof obligations]
        SOLVE((solve MCP Z3))
        OUT[cell outputs diagram IR proofs counterexamples]
        PORTS[cell-scoped ports spec_ir proof_report verified conformance_vectors]
        SRC --> PARSE --> IR --> PLAN --> SOLVE --> OUT --> PORTS
    end

    READY["`@branch READY
@when source_present and parser_pass and obligations_complete and solver_terminal and ports_published
@ensures DECISION_READY: decision = ready`"]

    BLOCKED["`@branch BLOCKED
@when not (source_present and parser_pass and obligations_complete and solver_terminal and ports_published)
@ensures DECISION_BLOCKED: decision = blocked`"]

    CHECK["`@verify ARCH_DETERMINISTIC: prove determinism
@verify ARCH_COVERAGE: prove partition_coverage
@verify ARCH_EXCLUSIVE: prove partition_exclusive
@verify READY_REACHABLE: witness branch READY
@verify BLOCKED_REACHABLE: witness branch BLOCKED`"]

    CTX --> SRC
    CTX --> READY --> CHECK
    CTX --> BLOCKED --> CHECK
    PORTS --> REPORT[downstream report and CI gate]

## 3. Core principles

1. **The notebook is the specification.** `.spec.ipynb` is authoritative;
   generated Markdown is a review/export artifact only.
2. **One cell is one self-verifying formal unit.** Its Mermaid source, rendered
   diagram, Canonical IR, obligations, and verdict share one cell identity.
3. **Mermaid declares the semantics.** Stable graph IDs plus `@type`,
   `@input`, `@output`, `@state-var`, `@state`, `@transition`, `@branch`,
   `@requires`, `@guard`, `@update`, `@when`, `@ensures`, `@invariant`,
   `@verify`, and `@witness` annotations are the only authored formal source.
4. **Canonical IR is generated and auditable.** It is the verifier input, never
   a second authoring surface. Every IR element maps back to a cell and source
   span.
5. **Z3 is reached only through `solve`.** B′ is preferred; `solve_smt` is the
   guarded escape hatch. The notebook never imports or invokes Z3 directly.
6. **Source is pure; outputs are derived.** Running a cell may emit diagram,
   IR, proofs, models, diagnostics, and ports, but may not modify source.
7. **Verification is fail-closed.** Parse/type errors, missing obligations,
   stale results, `unknown`, `timeout`, unsupported terms, and internal errors
   all block approval.
8. **Proof meaning is explicit.** Proof obligations use assert-negation
   (`unsat` proves no counterexample); witness obligations expect `sat`.
9. **Bounded is not proven.** Generated conformance vectors yield
   `conformance-passed`, never `refinement-proven`.
10. **Lowering is part of the trusted base.** An `unsat` result is proof-grade
    only when the NS-Mermaid→IR→B′/SMT lowering is validated.
11. **The verifier cannot certify itself.** A new or changed runtime profile is
    trusted only after native results differentially match independently authored
    direct `solve` fixtures and pass brain review.
12. **Approved semantics do not move during implementation repair.** Workers may
    repair code or adapters; a spec edit requires a new authoring approval and
    invalidates prior proof and conformance evidence.
13. **Agents never guess the dialect.** Every authoring task pins a versioned
    NS-Mermaid profile manifest. Compiler diagnostics, canonical round-trip,
    obligation completeness, and same-cell hash agreement gate acceptance.

## 4. Per-cell verification decision flow

Every `ns-mermaid` cell runs this pipeline as one execution. A cell-level report
is published only after all declared verification points reach terminal statuses.

> **Executable diagram:** the following native `ns_mermaid` cell preserves the
verification pipeline and proves the verified/failed decision partition.

The aggregate `verified` port is true only when parsing and typing succeeded,
all mandatory points were evaluated against the current source/IR hashes, and
each result matched its declared proof or witness expectation. Any source or
upstream-input edit immediately makes the prior output stale.

In [ ]:
flowchart TD
    CTX["`@spec NS-SPEC-PER-CELL-VERIFICATION-FLOW
@type VerificationDecision = enum[verified, failed]
@input mermaid_valid: Bool
@input types_valid: Bool
@input obligations_complete: Bool
@input lowering_supported: Bool
@input solver_expectations_match: Bool
@output decision: VerificationDecision`"]

    RUN([run ns-mermaid cell]) --> PARSE[parse annotations]
    PARSE --> TYPE[type IDs references expressions]
    TYPE --> IR[emit Canonical IR and source map]
    IR --> PLAN[generate named obligations]
    PLAN --> SOLVE((solve MCP Z3))
    SOLVE --> RESULT[attach proofs witnesses diagnostics]

    VERIFIED["`@branch VERIFIED
@when mermaid_valid and types_valid and obligations_complete and lowering_supported and solver_expectations_match
@ensures DECISION_VERIFIED: decision = verified`"]

    FAILED["`@branch FAILED
@when not (mermaid_valid and types_valid and obligations_complete and lowering_supported and solver_expectations_match)
@ensures DECISION_FAILED: decision = failed`"]

    CHECK["`@verify FLOW_DETERMINISTIC: prove determinism
@verify FLOW_COVERAGE: prove partition_coverage
@verify FLOW_EXCLUSIVE: prove partition_exclusive
@verify VERIFIED_REACHABLE: witness branch VERIFIED
@verify FAILED_REACHABLE: witness branch FAILED`"]

    CTX --> RUN
    CTX --> VERIFIED --> CHECK
    CTX --> FAILED --> CHECK
    RESULT --> AGG[aggregate proof_report and verified port]

## 4b. Determinism declared and verified in one cell

The v0.1 transfer contract guarded success on the output itself. In v0.3 the
same mistake is visible in its Mermaid declaration point:

> **Executable negative fixture:** the first native `ns_mermaid` cell after this
section preserves the output-dependent guards and is expected to return a
`DET_ORIGINAL` counterexample with `verified=false`.

Both `@when` clauses select an output value instead of partitioning inputs. The
generated determinism counterexample formula is `sat`, so the same cell reports
that one eligible input admits both unchanged/insufficient and transferred/success
results.

The corrected cell declares an input partition and its verification points:

> **Executable corrected fixture:** the second native `ns_mermaid` cell after
this section uses an input partition and explicit branch witnesses. All six
obligations must match.

One run of this cell renders the diagram and evaluates all six named points.
The expected outputs are `unsat` for determinism, gap, and overlap
counterexample formulas, and `sat` for each requested status witness. Each result
records the declaration cell version, IR hash, obligation hash, and exact
annotation source span.

In [ ]:
flowchart TD
    INPUT["`@spec TRANSFER-ORIGINAL
@type TransferStatus = enum[success, insufficient_balance]
@input source_balance: Int
@input target_balance: Int
@input amount: Int
@output status: TransferStatus
@output source_after: Int
@output target_after: Int
@requires PRE_SOURCE: source_balance >= 0
@requires PRE_TARGET: target_balance >= 0
@requires PRE_AMOUNT: amount > 0`"]

    INSUFFICIENT["`@branch INSUFFICIENT
@when status = insufficient_balance
@ensures STATUS_INSUFFICIENT: status = insufficient_balance
@ensures SOURCE_INSUFFICIENT: source_after = source_balance
@ensures TARGET_INSUFFICIENT: target_after = target_balance`"]

    SUCCESS["`@branch SUCCESS
@when status = success
@ensures STATUS_SUCCESS: status = success
@ensures SOURCE_SUCCESS: source_after = source_balance - amount
@ensures TARGET_SUCCESS: target_after = target_balance + amount`"]

    CHECK["`@verify DET_ORIGINAL: prove determinism`"]
    INPUT --> INSUFFICIENT --> CHECK
    INPUT --> SUCCESS --> CHECK
    CHECK --> V{{expected same-cell result: sat counterexample}}

In [ ]:
flowchart TD
    CTX["`@spec TRANSFER-001
@type TransferStatus = enum[success, invalid_amount, insufficient_balance]
@input source_balance: Int
@input target_balance: Int
@input amount: Int
@output status: TransferStatus
@output source_after: Int
@output target_after: Int
@requires PRE_SOURCE: source_balance >= 0
@requires PRE_TARGET: target_balance >= 0`"]

    INVALID["`@branch INVALID
@when amount <= 0
@ensures STATUS_INVALID: status = invalid_amount
@ensures SOURCE_INVALID: source_after = source_balance
@ensures TARGET_INVALID: target_after = target_balance`"]

    INSUFFICIENT["`@branch INSUFFICIENT
@when amount > 0 and source_balance < amount
@ensures STATUS_INSUFFICIENT: status = insufficient_balance
@ensures SOURCE_INSUFFICIENT: source_after = source_balance
@ensures TARGET_INSUFFICIENT: target_after = target_balance`"]

    SUCCESS["`@branch SUCCESS
@when amount > 0 and source_balance >= amount
@ensures STATUS_SUCCESS: status = success
@ensures SOURCE_SUCCESS: source_after = source_balance - amount
@ensures TARGET_SUCCESS: target_after = target_balance + amount`"]

    CHECK["`@invariant NONNEG: source_after >= 0 and target_after >= 0
@invariant CONSERVE: source_after + target_after = source_balance + target_balance
@verify DET_TRANSFER: prove determinism
@verify STATUS_GAP: prove partition_coverage
@verify STATUS_OVERLAP: prove partition_exclusive
@verify INVALID_REACHABLE: witness branch INVALID
@verify INSUFFICIENT_REACHABLE: witness branch INSUFFICIENT
@verify SUCCESS_REACHABLE: witness branch SUCCESS`"]

    CTX --> INVALID --> CHECK
    CTX --> INSUFFICIENT --> CHECK
    CTX --> SUCCESS --> CHECK

## 5. Verification menu — `@verify` points

| Declaration | Generated obligation | Expected status | Mismatch meaning |
|---|---|---|---|
| `@verify <point-id>: witness non_vacuity` | `SAT(assumptions ∧ requires)` | `sat` | `unsat` = vacuous cell |
| `@verify <point-id>: witness consistency` | `SAT(requires ∧ ensures ∧ invariants)` | `sat` | `unsat` = conflicting declarations |
| `@verify <point-id>: prove determinism` | `SAT(Pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2)` | `unsat` | `sat` = under-determined outputs |
| `@verify <point-id>: prove partition_coverage` | `SAT(pre ∧ ¬(g1 ∨ … ∨ gn))` | `unsat` | `sat` = uncovered input region |
| `@verify <point-id>: prove partition_exclusive` | `SAT(pre ∧ ⋁i<j(gi ∧ gj))` | `unsat` | `sat` = overlapping branches |
| `@verify <point-id>: witness branch <branch-id>` | `SAT(pre ∧ posts ∧ branch)` | `sat` | `unsat` = dead branch |
| `@verify <point-id>: prove initiate <inv>` | `SAT(initial ∧ ¬invariant)` | `unsat` | `sat` = invalid initial state |
| `@verify <point-id>: prove preserve <inv> on <transition>` | `SAT(inv ∧ guard ∧ update ∧ ¬inv(next))` | `unsat` | `sat` = transition counterexample |
| `@verify <point-id>: bounded_reachability k=<n>` | explicit k-step unroll | declared `sat` or `unsat` | always labeled bounded |

**Preservation is invariant-local.** `prove preserve I on T` assumes `I`, the
transition guard, and updates; it does not silently assume every other named
invariant. If a proof requires a stronger inductive hypothesis, encode it in
`I` or make the transition guard establish it.

**Constructive totality is a structural gate.** Every exhaustive input branch
must define every output with total, well-typed terms before solver execution.
Per-branch satisfiability is a reachability audit, not a proof of
`∀x∃y.Post(x,y)`.

The v0.3 expression grammar is QF_LIA + Bool + enum. Nonlinear multiplication,
division, modulo, unbounded quantifiers, and unsupported Mermaid constructs fail
with source-located diagnostics unless the verification point explicitly opts
into an allowed `solve_smt` theory. `unknown` and `timeout` never count as
proof.

## 6. Conformance generation from the same cell

Proof points audit the declaration; witness points can also emit bounded test
vectors. The `ns-mermaid` cell runner exposes these through a typed
`conformance_vectors` port without executing the implementation itself.

The following native `ns-mermaid` cell is the executable conformance pipeline. Its rendered flow preserves witness generation, proof publication, generated tests, and implementation comparison; its declarations make pass/block behavior solver-checkable.

Boundary requests are explicit annotations such as `@witness amount = 0`,
`@witness amount = 1`, or `@witness source_balance = amount - 1`. A plain `sat`
model is arbitrary and MUST NOT be described as minimized or boundary-selected.

`conformance-passed` means the implementation matched the declared outputs on
the emitted finite vectors. It is distinct from a proof. A false acceptance is
still possible if the NS-Mermaid parser, lowering, oracle generation, wrapper,
or implementation adapter is wrong; those components require differential and
end-to-end tests.

In [ ]:
flowchart LR
    CELL["`@spec NS-SPEC-CONFORMANCE-PIPELINE
@type ConformanceDecision = enum[conformance_passed, blocked]
@input proof_verified: Bool
@input vectors_present: Bool
@input adapter_ready: Bool
@input implementation_match: Bool
@output decision: ConformanceDecision`"]

    IR[Canonical IR]
    W1["witness each status"]
    W2["witness boundary predicates"]
    W3["witness invariant edges"]
    SOLVE((solve MCP))
    OUT[proof report plus vectors]
    PORT[conformance_vectors port]
    TEST[language-specific generated tests]
    IMPL[real implementation]

    PASS["`@branch PASS
@when proof_verified and vectors_present and adapter_ready and implementation_match
@ensures PASS_DECISION: decision = conformance_passed`"]

    BLOCKED["`@branch BLOCKED
@when not (proof_verified and vectors_present and adapter_ready and implementation_match)
@ensures BLOCK_DECISION: decision = blocked`"]

    CHECK["`@verify CONFORMANCE_DETERMINISTIC: prove determinism
@verify CONFORMANCE_COMPLETE: prove partition_coverage
@verify CONFORMANCE_EXCLUSIVE: prove partition_exclusive
@verify PASS_REACHABLE: witness branch PASS
@verify BLOCK_REACHABLE: witness branch BLOCKED`"]

    CELL --> IR
    IR --> W1 --> SOLVE
    IR --> W2 --> SOLVE
    IR --> W3 --> SOLVE
    SOLVE --> OUT --> PORT --> TEST --> IMPL
    IMPL --> PASS --> CHECK
    IMPL --> BLOCKED --> CHECK

## 7. Same-cell `solve` integration and ports

The following native `ns-mermaid` cell makes the solve sequence executable as a verification-and-publication gate. The rendered pipeline retains authoring, normalization, per-obligation solving, same-cell outputs, typed-port publication, and downstream consumption.

### Required output bundle

- `diagram`: rendered Mermaid or a source-located render error.
- `spec_ir`: normalized Canonical IR plus schema version and IR hash.
- `obligations[]`: ID, kind, expected status, lowered form, and obligation hash.
- `results[]`: solver status, model/counterexample, duration, and optional
  repository-local `solve_id`.
- `proof_report`: aggregate pass/fail/inconclusive state.
- `verified`: true only for a fresh, complete, matching proof report.
- `conformance_vectors`: requested bounded witnesses, if any.
- `diagnostics`: phase, stable code, annotation/node ID, source span, and repairability.

Canonical hashes cover schema-versioned semantic content only. Transient duration,
transport metadata, and `solve_id` are displayed but excluded, so UI and headless
runs can produce identical semantic hashes.

The cell source and outputs share one notebook cell ID but remain separate data.
The runner emits five cell-scoped ports—`spec_ir`, `proof_report`, `verified`,
`conformance_vectors`, and `diagnostics`—under a producer namespace derived from
that stable cell identity; a second NS-Mermaid cell may reuse local
annotation IDs without sharing producer names. The runner MUST NOT write source
or consume its own ports. Persistence through `solve_id` is a handoff cache, not
the durable source of truth—the declaration, normalized IR hash, and proof
report stay in the notebook artifact.

Every consumer MUST bind a port value to its proof identity tuple
`(cell_id, cell_version, source_hash, ir_hash, report_hash)`. Port version
movement alone is not semantic evidence and cannot make a stale proof current.

In [ ]:
flowchart LR
    AUTHOR["`@spec NS-SPEC-SOLVE-SEQUENCE
@type PublishDecision = enum[publish, block]
@input source_valid: Bool
@input typing_valid: Bool
@input obligations_complete: Bool
@input solver_terminal: Bool
@input hashes_match: Bool
@input ports_published: Bool
@output decision: PublishDecision`"]

    RUNNER[parse type-check normalize and hash]
    SOLVE((solve each named obligation))
    OUTPUTS[diagram IR proof report diagnostics]
    PORTS[typed ports plus proof identity]
    DOWNSTREAM[downstream report or CI]

    PUBLISH["`@branch PUBLISH
@when source_valid and typing_valid and obligations_complete and solver_terminal and hashes_match and ports_published
@ensures PUBLISH_DECISION: decision = publish`"]

    BLOCK["`@branch BLOCK
@when not (source_valid and typing_valid and obligations_complete and solver_terminal and hashes_match and ports_published)
@ensures BLOCK_DECISION: decision = block`"]

    CHECK["`@verify PUBLISH_DETERMINISTIC: prove determinism
@verify PUBLISH_COMPLETE: prove partition_coverage
@verify PUBLISH_EXCLUSIVE: prove partition_exclusive
@verify PUBLISH_REACHABLE: witness branch PUBLISH
@verify BLOCK_REACHABLE: witness branch BLOCK`"]

    AUTHOR --> RUNNER --> SOLVE --> OUTPUTS --> PORTS --> DOWNSTREAM
    DOWNSTREAM --> PUBLISH --> CHECK
    DOWNSTREAM --> BLOCK --> CHECK

## 8. Cell and notebook lifecycle

The following native `ns-mermaid` state cell is the executable lifecycle. It retains the complete status topology and proves that approval is possible only for a fresh, complete proof. All fifteen rendered edges use stable `LifecycleEvent` enum labels and exact `@event`/`@from`/`@to` bindings; initiation and preservation cover type-check, verification outcomes, approval, implementation entry, conformance/refinement completion, invalidation, edit, and rerun.

A notebook-level gate is the conjunction of its required cell gates. Every
formal cell must be fresh; optional explanatory Markdown cells do not
participate. Changing one `ns-mermaid` cell invalidates that cell and only its
reactive downstream consumers. Approval records the cell source hash, IR hash,
obligation hashes, and aggregate report hash.

### 8.1 Confirmed runtime status and remaining lifecycle gaps

Notebook MCP confirmation on 2026-08-01 established that multiple native cells
coexist, publish cell-namespaced ports, preserve proof identity across reopen,
and isolate their source/IR/report hashes on rerun. The lifecycle diagram above
remains normative rather than fully implemented, because three gaps are still
observable:

1. after editing one cell, its prior proof output remains visible until rerun and
   the DAG reports no stale cell even though the source cell version and proof
   `cell_version` differ;
2. the edit advances every existing NS-Mermaid `verified` port version rather
   than only the edited cell's port and reactive downstream consumers;
3. reopening and rerunning a notebook created by the pre-namespacing runtime can
   leave legacy global `spec_ir`, `proof_report`, `verified`, and
   `conformance_vectors` aliases visible beside the new cell-scoped ports.

Until these gaps are fixed, a brain or CI gate MUST compare the current cell ID,
cell version, source hash, IR hash, and report hash directly. A visible
`verified: true` value or an empty DAG stale list is insufficient approval
evidence.

In [ ]:
stateDiagram-v2
    [*] --> DRAFT
    DRAFT --> INVALID: PARSE_ERROR
    DRAFT --> TYPED: TYPE_PASS
    TYPED --> VERIFYING: START_VERIFY
    VERIFYING --> VERIFIED: VERIFY_OK
    VERIFYING --> FAILED: VERIFY_FAIL
    VERIFYING --> INCONCLUSIVE: VERIFY_INCONCLUSIVE
    VERIFIED --> APPROVED: APPROVE
    APPROVED --> IMPLEMENTATION_PENDING: START_IMPLEMENTATION
    IMPLEMENTATION_PENDING --> CONFORMANCE_PASSED: CONFORMANCE_PASS
    IMPLEMENTATION_PENDING --> REFINEMENT_PROVEN: REFINEMENT_PASS
    VERIFIED --> STALE: VERIFIED_INVALIDATED
    FAILED --> STALE: FAILED_INVALIDATED
    INCONCLUSIVE --> STALE: INCONCLUSIVE_INVALIDATED
    APPROVED --> STALE: EDIT
    STALE --> TYPED: RERUN

    note right of DRAFT
      @spec NS-SPEC-CELL-LIFECYCLE
      @type LifecycleEvent = enum[PARSE_ERROR, TYPE_PASS, START_VERIFY, VERIFY_OK, VERIFY_FAIL, VERIFY_INCONCLUSIVE, APPROVE, START_IMPLEMENTATION, CONFORMANCE_PASS, REFINEMENT_PASS, VERIFIED_INVALIDATED, FAILED_INVALIDATED, INCONCLUSIVE_INVALIDATED, EDIT, RERUN]
      @input event: LifecycleEvent
      @state-var fresh: Bool
      @state-var proof_complete: Bool
      @state-var approved: Bool
      @requires INIT_FRESH: fresh
      @requires INIT_INCOMPLETE: not proof_complete
      @requires INIT_UNAPPROVED: not approved
      @state DRAFT
      @state INVALID
      @state TYPED
      @state VERIFYING
      @state VERIFIED
      @state FAILED
      @state INCONCLUSIVE
      @state APPROVED
      @state IMPLEMENTATION_PENDING
      @state CONFORMANCE_PASSED
      @state REFINEMENT_PROVEN
      @state STALE
      @invariant APPROVAL_VALID: not approved or (fresh and proof_complete)
      @verify INIT_APPROVAL_VALID: prove initiate APPROVAL_VALID
    end note

    note right of DRAFT
      @transition TYPE_PASS
      @event event = TYPE_PASS
      @from DRAFT
      @to TYPED
      @guard fresh and not proof_complete
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = approved
      @verify TYPE_PASS_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on TYPE_PASS
    end note

    note right of TYPED
      @transition START_VERIFY
      @event event = START_VERIFY
      @from TYPED
      @to VERIFYING
      @guard fresh and not proof_complete
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = approved
      @verify START_VERIFY_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on START_VERIFY
    end note

    note right of VERIFYING
      @transition VERIFY_OK
      @event event = VERIFY_OK
      @from VERIFYING
      @to VERIFIED
      @guard fresh and not proof_complete
      @update fresh' = fresh
      @update proof_complete' = true
      @update approved' = approved
      @verify VERIFY_OK_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on VERIFY_OK
    end note

    note right of VERIFIED
      @transition APPROVE
      @event event = APPROVE
      @from VERIFIED
      @to APPROVED
      @guard fresh and proof_complete
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = true
      @verify APPROVE_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on APPROVE
    end note

    note right of APPROVED
      @transition EDIT
      @event event = EDIT
      @from APPROVED
      @to STALE
      @guard approved
      @update fresh' = false
      @update proof_complete' = false
      @update approved' = false
      @verify EDIT_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on EDIT
    end note

    note right of STALE
      @transition RERUN
      @event event = RERUN
      @from STALE
      @to TYPED
      @guard not fresh
      @update fresh' = true
      @update proof_complete' = false
      @update approved' = false
      @verify RERUN_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on RERUN
    end note

    note right of DRAFT
      @transition PARSE_ERROR
      @event event = PARSE_ERROR
      @from DRAFT
      @to INVALID
      @guard fresh and not proof_complete
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = approved
      @verify PARSE_ERROR_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on PARSE_ERROR
    end note

    note right of VERIFYING
      @transition VERIFY_FAIL
      @event event = VERIFY_FAIL
      @from VERIFYING
      @to FAILED
      @guard fresh and not proof_complete
      @update fresh' = fresh
      @update proof_complete' = false
      @update approved' = false
      @verify VERIFY_FAIL_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on VERIFY_FAIL
    end note

    note right of VERIFYING
      @transition VERIFY_INCONCLUSIVE
      @event event = VERIFY_INCONCLUSIVE
      @from VERIFYING
      @to INCONCLUSIVE
      @guard fresh and not proof_complete
      @update fresh' = fresh
      @update proof_complete' = false
      @update approved' = false
      @verify VERIFY_INCONCLUSIVE_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on VERIFY_INCONCLUSIVE
    end note

    note right of APPROVED
      @transition START_IMPLEMENTATION
      @event event = START_IMPLEMENTATION
      @from APPROVED
      @to IMPLEMENTATION_PENDING
      @guard approved
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = approved
      @verify START_IMPLEMENTATION_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on START_IMPLEMENTATION
    end note

    note right of IMPLEMENTATION_PENDING
      @transition CONFORMANCE_PASS
      @event event = CONFORMANCE_PASS
      @from IMPLEMENTATION_PENDING
      @to CONFORMANCE_PASSED
      @guard approved
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = approved
      @verify CONFORMANCE_PASS_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on CONFORMANCE_PASS
    end note

    note right of IMPLEMENTATION_PENDING
      @transition REFINEMENT_PASS
      @event event = REFINEMENT_PASS
      @from IMPLEMENTATION_PENDING
      @to REFINEMENT_PROVEN
      @guard approved
      @update fresh' = fresh
      @update proof_complete' = proof_complete
      @update approved' = approved
      @verify REFINEMENT_PASS_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on REFINEMENT_PASS
    end note

    note right of VERIFIED
      @transition VERIFIED_INVALIDATED
      @event event = VERIFIED_INVALIDATED
      @from VERIFIED
      @to STALE
      @guard fresh and proof_complete
      @update fresh' = false
      @update proof_complete' = false
      @update approved' = false
      @verify VERIFIED_INVALIDATED_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on VERIFIED_INVALIDATED
    end note

    note right of FAILED
      @transition FAILED_INVALIDATED
      @event event = FAILED_INVALIDATED
      @from FAILED
      @to STALE
      @guard not approved
      @update fresh' = false
      @update proof_complete' = false
      @update approved' = false
      @verify FAILED_INVALIDATED_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on FAILED_INVALIDATED
    end note

    note right of INCONCLUSIVE
      @transition INCONCLUSIVE_INVALIDATED
      @event event = INCONCLUSIVE_INVALIDATED
      @from INCONCLUSIVE
      @to STALE
      @guard not approved
      @update fresh' = false
      @update proof_complete' = false
      @update approved' = false
      @verify INCONCLUSIVE_INVALIDATED_PRESERVES_APPROVAL: prove preserve APPROVAL_VALID on INCONCLUSIVE_INVALIDATED
    end note

## 9. NS-Mermaid profile and milestones

### 9.1 Normative declaration vocabulary

| Annotation | Meaning |
|---|---|
| `@spec <id>` | begins one cell-local specification namespace |
| `@type <name> = <type-expr>` | declares an alias over `Int`, `Bool`, or `enum[...]` |
| `@input <name>: <type>` | declares a universally quantified logical input |
| `@output <name>: <type>` | declares a result related to the logical inputs |
| `@state-var <name>: <type>` | declares mutable model state for transition systems |
| `@requires <id>: <expr>` | input assumption or precondition |
| `@state <id>` / Mermaid node ID | declares a stable control-state identity |
| `@transition <id>` | binds a stable transition ID to diagram topology |
| `@from <state-id>` | explicitly binds a transition source state |
| `@to <state-id>` | explicitly binds a transition destination state |
| `@event event = <enum-member>` | binds a typed event predicate to the exact rendered transition label |
| `@branch <id>` | binds a stable decision-branch ID to a Mermaid node |
| `@guard <expr>` | transition enabling predicate |
| `@update <state-var>' = <expr>` | next-state assignment |
| `@when <expr>` | input branch predicate |
| `@ensures <id>: <expr>` | output relation or postcondition |
| `@invariant <id>: <expr>` | named state/output invariant |
| `@verify <id>: <kind>` | names a proof or witness obligation |
| `@witness <expr>` | adds an explicit bounded-model predicate |

Annotations live visibly inside Mermaid Markdown-string node labels. The parser
reads source text, not rendered DOM. IDs are case-sensitive and unique within a
cell. References must resolve locally or through explicit typed input ports;
implicit notebook-global symbols are forbidden.

For `stateDiagram-v2`, the shared declaration note defines an event enum and
`@input event: <EventType>`. Every rendered transition edge has a stable label
that is an enum member and exactly one formal transition note containing
`@transition`, `@event event = <member>`, `@from`, and `@to`. One Mermaid note
may declare at most one transition. Complete one-to-one edge/event binding is
mandatory: an unbound rendered transition makes `binding_complete=false` and
prevents aggregate verification even when every executed solver obligation
matches.

### 9.2 Revised milestones

1. **NS-Mermaid grammar and parser** — supported Mermaid subset, annotation
   lexer, stable IDs, source spans, expression grammar, and diagnostics.
2. **Native `ns-mermaid` cell runner** — kernel-independent execution, Mermaid
   rendering, same-cell outputs, staleness, cancellation, and typed ports.
3. **Canonical IR and trusted lowering** — normalization plus B′/SMT generation,
   independent direct-`solve` fixtures, differential checks, round-trip checks,
   and mutation tests against false `unsat` results.
4. **Verification planner** — named proof/witness expansion, expectation
   matching, counterexample mapping, and aggregate cell reports.
5. **Conformance vector emitter** — explicit boundaries and witnesses into
   language-specific black-box tests.
6. **Headless notebook gate** — deterministic reactive execution, output-hash
   validation, CI reporting, and approval records.
7. **Agent correction and trust gate** — bootstrap evidence packets,
   profile-scoped differential approval, immutable approved specs,
   conformance-driven implementation repair, and brain-side independent reruns.
8. **Compiler-backed agent authoring kit** — versioned profile manifests,
   constrained scaffolds, source-located diagnostics, canonical formatting,
   obligation-completeness checks, and Notebook MCP same-cell hash comparison.

Markdown `.spec.md` becomes a generated snapshot. It may show the Mermaid source,
normalized IR, and proof report, but editing it never changes the authoritative
notebook.

## 10. Solver-substrate evidence and verifier caveats

These runs validate the proof obligations and existing `solve` substrate; they
predate and therefore do not validate the native `ns-mermaid` cell runner.

All runs were executed on 2026-07-29 through the existing `solve_constraints`
MCP tool using B′ only. Runs #2–8 included the transfer input requirements,
postcondition implications, non-negative output invariants, and balance
conservation; run #1 intentionally encoded a mutant that violates them. `sat`
models below are concrete witnesses, not golden assignments;
Z3 may return different witnesses for the same satisfiable predicate. Transient
wall-clock timings are intentionally omitted.

| # | Check | Encoding (B′) | Reproduced result |
|---|---|---|---|
| 1 | Seeded mutation witness | insufficient input plus a mutant that returns `success` and applies the transfer | **sat** → `src=6,tgt=0,amt=8,status=success,sa=-2,ta=8` |
| 2 | **Determinism, ORIGINAL contract** | `pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2` | **sat** → at `src=2,tgt=1,amt=1`, `insufficient`/unchanged and `success`/transferred are both valid |
| 3 | **Determinism, CORRECTED contract** | same counterexample formula with input-guarded posts | **unsat** → no two distinct outputs exist |
| 4 | Status-partition gap | `pre ∧ ¬(g_invalid ∨ g_insufficient ∨ g_success)` | **unsat** → guards are exhaustive |
| 5 | Status-partition overlap | `pre ∧ ⋁i<j(gi ∧ gj)` | **unsat** → guards are mutually exclusive |
| 6 | Conformance witness — `success` | `pre ∧ corrected_contract ∧ status=success` | **sat** → `src=7,tgt=7,amt=5 ⇒ sa=2,ta=12` |
| 7 | Conformance witness — `invalid_amount` | `pre ∧ corrected_contract ∧ status=invalid_amount` | **sat** → `src=3,tgt=0,amt=-8 ⇒ unchanged` |
| 8 | Conformance witness — `insufficient_balance` | `pre ∧ corrected_contract ∧ status=insufficient_balance` | **sat** → `src=3,tgt=0,amt=8 ⇒ unchanged` |
| 9 | SHIP preservation, original guard | `inv ∧ status=paid ∧ ship_update ∧ ¬inv(next)` | **sat** → `captured=0,next_status=shipped` |
| 10 | SHIP preservation, strengthened guard | run #9 plus `captured>0` | **unsat** → strengthened transition is inductive |

### What this proves

- Runs #2–3 reproduce the determinism defect and prove the corrected encoded
  relation deterministic via the assert-negation pattern.
- Runs #4–5 separately prove status-partition coverage and exclusivity; neither
  result is a substitute for full relation totality.
- Runs #6–8 establish that every declared transfer status has at least one
  witness suitable for a generated test. They remain bounded examples, not a
  refinement proof.
- Run #1 is an explicit seeded mutation, not symbolic inspection of a real
  implementation. It demonstrates a test-generation target without reviving the
  deleted implementation adapter.
- Runs #9–10 reproduce the distinction between invariant preservation and
  reachability discussed in §10.2 C2.

### Bottom line

The solver results validate the logical correction and the existing `solve` MCP
integration. Those historical runs do **not** validate the native NS-Mermaid cell
runner, its lowering, or the generated-test runner. Section 13 adds scoped
relational Boolean/enum differential evidence; broader profiles still require
their own property-preserving and end-to-end conformance tests.

---

### 10.1 Grounding against `crates/spur-solver`

The solver-integration claims were checked against the real solver source via code-explore, and the unsat-core gap was confirmed by execution:

| Claim | Source | Status |
|---|---|---|
| B′ `Variable` / `ConstraintExpr` / `ConstraintOp` grammar (vars, tagged expr, ops, no `div`) | `spur-solver/src/types.rs:30-179` | exact match |
| Status envelope `Sat` / `Unsat` / `Unknown` / `Timeout` (+`Error`) | `types.rs:320-334` | match |
| Encoder hardcodes `(set-logic QF_NIA)` — nonlinear `mul` supported | `encode.rs:125` | reframes QF_LIA as a self-imposed fragment |
| SMT gate allowlist = `set-logic` / `assert` / `check-sat` / `get-model` / `get-value` / `push` / `pop` / `declare-*` | `smt_gate.rs:143-153` | **`get-unsat-core` REJECTED** → client-side subset-iteration |
| `unknown` never collapsed to `unsat` | test `fake_solver_unknown_is_not_collapsed_into_unsat` | enforced in code |
| `persist` → `solve_id`, worker-side solve tools | `spur-core/src/worker_server.rs`, test `solve_constraints_persists_when_requested` | wired + tested |

The `get-unsat-core` rejection was reproduced live: a `solve_smt` script containing it returns `command get-unsat-core at byte 71 is not allowed`. Net: the v0.3 cell runner can remain a **pure client of the existing `solve` MCP with zero changes to `spur-solver`**; widening the gate for native unsat-cores is a future optimization, not a requirement.

---

### 10.2 Verify-the-verifier corrections

**C1. The lowering is the #1 soundness risk (false unsat).** During evaluation, a mis-nested disjunction in a B-prime encoding produced three spurious `unsat` results — each indistinguishable from a genuine proof. The solver was correct (a minimal `sat` probe confirmed the engine and enum encoding were flawless); the bug was purely in the lowering. Lesson: a verdict is only as sound as the NS-Mermaid→IR→B′ lowering layer. The rule `unknown is not success` must extend to: an `unsat` produced from an untrusted lowering is not a proof until the lowering itself is validated — a lowering bug silently mints false proofs, strictly worse than `unknown`. Implication: the lowering layer needs round-trip or property-preserving checks (or a small trusted kernel); this is the top engineering risk for an NS-Spec implementation, ahead of headless-CI and executable-cell security.

**C2. Label §16.7 preservation vs §16.8 reachability distinctly (SHIP downgrade).** The §16.7 invariant-preservation check runs over all invariant-satisfying states, not just reachable ones. For ORDER-LIFECYCLE it returns `sat` (counterexample `captured=0`): `INV-SHIPPED-PAID` is non-inductive, because SHIP from `paid` does not require `captured>0`. This is a code smell, not a runtime defect — the violating state `paid` with `captured=0` is unreachable (the only transition into `paid` is PAY, which sets `captured=amount>0`). §16.7 and §16.8 return different verdicts by design; a spec can fail §16.7 yet be safe. Rule: the proof report MUST label which check produced a verdict, and never present a §16.7 `sat` as unsafe without a reachability verdict. (Reachability here is by inspection of the transition relation; a solver proof needs a correct unrolling — which is easy to mis-encode, see C1.)

---

## 11. Normative v0.3 decision — one Mermaid declaration, one cell, one verdict

**Decision.** The authoritative spec format is `.spec.ipynb`. Each formal cell
has native type `ns-mermaid`. Its single source is annotated Mermaid and its
single execution performs both declaration processing and verification through
`solve`. The same cell owns the rendered diagram, normalized IR, named proof
results, counterexamples, and typed output ports.

> **Executable diagram:** the following native `ns_mermaid` cell formalizes the
one-source/one-run/one-verdict authority boundary and proves its accept/reject
partition.

### Same-cell means source plus outputs, not self-mutation

A notebook code cell already separates immutable source from derived outputs.
`ns-mermaid` uses that model without a general-purpose language kernel. The
runner reads its source and explicit upstream ports, then replaces only its
outputs. It never writes source, reads its prior output as an input, or mutates a
cell it depends on. Consequently, one cell can self-verify without a graph
feedback edge.

### Sole authored declaration language

There is no separate typed expression block and no hand-authored Z3 cell.
`@type`, `@input`, `@output`, `@state-var`, `@state`, `@transition`,
`@branch`, `@requires`, `@guard`, `@update`, `@when`, `@ensures`,
`@invariant`, `@verify`, and `@witness` annotations are part of the strict
NS-Mermaid source.
Canonical IR is a generated, auditable output. Z3 sees only the B′/SMT lowering
of that IR through the existing `solve` service.

Generated `.spec.md` is a portable review snapshot, not an independently
editable profile. Import or round-trip authoring is outside the v0.3 MVP because
it would reintroduce two sources of truth.

In [ ]:
flowchart TD
    CTX["`@spec NS-SPEC-ONE-CELL-AUTHORITY
@type AuthorityDecision = enum[accept, reject]
@input source_present: Bool
@input parse_type_pass: Bool
@input obligations_complete: Bool
@input solver_terminal: Bool
@input proof_identity_match: Bool
@output decision: AuthorityDecision`"]

    subgraph ONE[one ns-mermaid notebook cell]
        SOURCE[annotated Mermaid source]
        RUNNER[parse type normalize hash and plan]
        SOLVE((solve MCP Z3))
        OUTPUTS[diagram IR proofs counterexamples and ports]
        SOURCE --> RUNNER --> SOLVE --> OUTPUTS
    end

    ACCEPT["`@branch ACCEPT
@when source_present and parse_type_pass and obligations_complete and solver_terminal and proof_identity_match
@ensures DECISION_ACCEPT: decision = accept`"]

    REJECT["`@branch REJECT
@when not (source_present and parse_type_pass and obligations_complete and solver_terminal and proof_identity_match)
@ensures DECISION_REJECT: decision = reject`"]

    CHECK["`@verify AUTHORITY_DETERMINISTIC: prove determinism
@verify AUTHORITY_COVERAGE: prove partition_coverage
@verify AUTHORITY_EXCLUSIVE: prove partition_exclusive
@verify ACCEPT_REACHABLE: witness branch ACCEPT
@verify REJECT_REACHABLE: witness branch REJECT`"]

    CTX --> SOURCE
    CTX --> ACCEPT --> CHECK
    CTX --> REJECT --> CHECK
    OUTPUTS --> DOWNSTREAM[report conformance and CI cells]

### 11.1 Cell authoring and execution procedure

1. **Create one `ns-mermaid` cell** and assign a stable cell ID.
2. **Declare the visual topology** with explicit Mermaid node IDs and unique
   transition/branch labels.
3. **Declare formal meaning in the same source** using visible `@type`,
   `@input`, `@output`, `@state-var`, control IDs, predicates, updates,
   postconditions, and invariants.
4. **Name every required check** with `@verify`; add explicit `@witness`
   predicates for boundary or conformance models.
5. **Run that same cell.** The native runner parses Mermaid and annotations,
   resolves references, type-checks expressions, and records source spans.
6. **Inspect normalized IR output.** Verification cannot be approved until the
   human-visible IR matches the intended diagram semantics.
7. **Generate obligations.** Each named point receives an expectation, lowered
   expression, source hash, IR hash, and obligation hash.
8. **Verify through `solve`.** B′ goes to `solve_constraints`; an explicitly
   allowed theory goes to `solve_smt`. The notebook never invokes Z3 directly.
9. **Render same-cell results.** Proof badges appear next to declaration points;
   `sat` proof failures include counterexamples mapped back to annotations.
10. **Emit typed ports.** Fresh IR, proof report, verified state, and optional
    conformance vectors flow to downstream report/CI cells.
11. **Invalidate on change.** Editing source or an upstream typed input marks all
    prior outputs stale before reactive re-execution.
12. **Approve or block.** Approval requires a fresh normalized IR review and all
    mandatory point expectations to match.

### 11.2 Failure semantics

| Condition | Same-cell outcome |
|---|---|
| Mermaid render error | source-located parse failure; no solver call |
| Valid Mermaid but invalid NS annotation | profile diagnostic; no solver call |
| Unresolved ID or type mismatch | typed diagnostic; no solver call |
| Unsupported expression/theory | `unsupported-term`; fail closed |
| Proof obligation returns `sat` | refuted with concrete counterexample |
| Witness obligation returns `unsat` | requested branch/model is unreachable |
| `unknown`, `timeout`, or solver error | inconclusive; fail closed |
| Source/IR/obligation hash mismatch | stale; result cannot approve |
| Some points pass and one fails | retain all point results; aggregate fails |
| Cell run is cancelled | partial results marked incomplete and non-approving |

### 11.3 Adoption conditions and test strategy

1. **Native constrained runner.** Add an `ns-mermaid` cell type rather than
   executing Python or arbitrary JavaScript. Enforce source-size, AST-size,
   obligation-count, solver-time, and output-size limits.
2. **Headless parity.** The same cell runner and reactive semantics must execute
   without a window for CI; UI and headless runs must produce identical hashes.
3. **Trusted lowering evidence.** Golden parser fixtures, source-span tests,
   normalized-IR snapshots, B′/SMT differential tests, and mutation tests must
   detect dropped, inverted, duplicated, or mis-nested clauses.
4. **Same-cell integration tests.** Editing a declaration must stale its outputs,
   rerun its proofs, replace only outputs, and rerun only downstream consumers.
5. **Status tests.** Exercise `sat`, `unsat`, `unknown`, `timeout`, unsupported,
   cancellation, mixed multi-point results, and counterexample source mapping.
6. **Security tests.** Mermaid and annotation input must never escape into shell,
   Python, raw Z3 argv, filesystem access, or uncontrolled network calls.
7. **Artifact tests.** Saving/reopening preserves source and outputs; approval is
   rejected when persisted hashes no longer match current source.
8. **Agent correction tests.** Bootstrap workers must compare native results with
   independent direct-`solve` fixtures; implementation workers must be unable to
   release after changing an approved specification hash.
9. **Agent authoring tests.** Prompt-only and persistent-AST-first submissions
   must be rejected; scaffolded NS-Mermaid must pass profile binding, parse/type,
   canonical round-trip, obligation-completeness, same-cell, and hash gates.

**Top risk: lowering soundness.** A buggy NS-Mermaid→IR→B′/SMT lowering can mint
false `unsat` proofs. The lowering belongs to the trusted base and must be small,
versioned, auditable, and tested independently of Z3.

`ns-spec.spec.ipynb` remains useful prototype evidence for the logical examples,
but its hand-authored Python/Z3 cells are **not** a conforming v0.3 reference.
The native acceptance-gate cell in §13 is the first real `ns_mermaid` reference
and supplies relational Boolean/enum bootstrap evidence. A full reference still
requires cell-local ports, structured source diagnostics, complete proof identity,
and state-machine profile cells.

---

## 12. Brain/worker self-correction — staged trust, not self-certification

**Decision.** SPUR adopts both correction loops in strict order:

1. **Runtime bootstrap:** brain and workers implement and repair the
   `ns-mermaid` parser, IR, lowering, solver integration, outputs, and ports
   against an independent oracle.
2. **Application repair:** only after the runtime crosses the trust gate may
   workers use approved `ns-mermaid` cells to repair business-code
   implementations.

The native verifier MUST NOT establish its own trust merely by returning
`verified=true`. During bootstrap, its verdict is evidence under test.

### 12.1 Solver evaluation of the staged design

The loop gates were encoded in B′ and evaluated through `solve_constraints` on
2026-07-30. These results prove consistency of the stated Boolean/enum policy;
they do not replace parser, lowering, integration, or security tests.

| Query | Result | Interpretation |
|---|---|---|
| Require both a trusted runtime and releasable application | `sat`, model selects `staged_both` | the two-loop architecture is feasible |
| Release runtime while any oracle, fixture, differential, or brain-review gate is absent | `unsat` | unsafe runtime release is excluded by the proposed gate |
| Release application while any trust, approval, hash, native-proof, conformance, or brain-review gate is absent | `unsat` | unsafe application release is excluded by the proposed gate |
| Trust native runtime from its own passing verdict while no independent oracle exists | `sat` counterexample | naïve self-certification is unsafe |
| Release after native proof and conformance pass while the approved spec hash changed | `sat` counterexample | naïve repair permits goalpost shifting |

### 12.2 Runtime-bootstrap correction loop

The following native `ns-mermaid` cell is the executable runtime-bootstrap trust gate. It preserves the fixture/native/oracle/differential/repair loop and proves that profile trust is granted exactly when every independent gate passes.

The independent oracle is a checked-in fixture plus a direct
`solve_constraints` or guarded `solve_smt` encoding that does not traverse the
native NS-Mermaid lowering being tested. Its source and expected-result hashes
are approved before the runtime patch begins. A worker MUST NOT change both the
runtime and its oracle in one task; oracle changes require a separate review and
re-bootstrap every affected profile. Matching one fixture establishes trust only
for that covered grammar and obligation profile; unsupported profiles remain
untrusted.

In [ ]:
flowchart LR
    FIXTURE["`@spec NS-SPEC-RUNTIME-TRUST-GATE
@type TrustDecision = enum[trusted, untrusted]
@input profile_pinned: Bool
@input fixture_approved: Bool
@input oracle_independent: Bool
@input differential_match: Bool
@input brain_review_pass: Bool
@output decision: TrustDecision`"]

    NATIVE[native ns-mermaid runner under test]
    ORACLE[independent direct solve encoding]
    ACTUAL[actual IR obligations verdict and ports]
    EXPECTED[expected status witness or counterexample]
    DIFF{differential comparison}
    REPAIR[worker repairs parser planner lowering or integration]
    REVIEW[brain independently reruns fixture and oracle]

    TRUSTED["`@branch TRUSTED
@when profile_pinned and fixture_approved and oracle_independent and differential_match and brain_review_pass
@ensures TRUST_DECISION: decision = trusted`"]

    UNTRUSTED["`@branch UNTRUSTED
@when not (profile_pinned and fixture_approved and oracle_independent and differential_match and brain_review_pass)
@ensures UNTRUST_DECISION: decision = untrusted`"]

    CHECK["`@verify TRUST_DETERMINISTIC: prove determinism
@verify TRUST_COMPLETE: prove partition_coverage
@verify TRUST_EXCLUSIVE: prove partition_exclusive
@verify TRUST_REACHABLE: witness branch TRUSTED
@verify UNTRUST_REACHABLE: witness branch UNTRUSTED`"]

    FIXTURE --> NATIVE --> ACTUAL --> DIFF
    FIXTURE --> ORACLE --> EXPECTED --> DIFF
    DIFF --> REVIEW
    REVIEW --> TRUSTED --> CHECK
    REVIEW --> UNTRUSTED --> CHECK
    UNTRUSTED --> REPAIR --> NATIVE

### 12.3 Repair routing and worker evidence

The phase and evidence determine which artifact a worker may change:

| Evidence | Bootstrap worker action | Spec-authoring action | Implementation-worker action |
|---|---|---|---|
| Mermaid or annotation parse differs from fixture | repair parser/diagnostics | repair cell source | block and escalate; approved source is immutable |
| Canonical IR differs from golden IR | repair parsing/normalization | inspect intended semantics, then request approval | block and escalate |
| IR matches but native result differs from direct `solve` | repair obligation planner or lowering | no spec edit | no implementation edit |
| Native and direct solve agree on a proof counterexample | fixture/spec defect candidate | repair spec, then obtain new approval | block and return to spec-authoring state |
| Native proof passes but conformance fails | no runtime edit unless oracle mapping differs | no spec edit | repair implementation or adapter |
| `unknown`, timeout, cancellation, unsupported, or internal error | simplify, diagnose, or escalate | never approve | never release |
| Ports, staleness, or cascade differ from fixture | repair notebook cell integration | no semantic edit | no implementation edit |

Every bootstrap worker completion MUST include:

- fixture and declaration cell IDs;
- cell source hash and expected/actual IR hashes;
- expected obligation statuses and independent `solve_id` values when persisted;
- native obligation results and differential verdict;
- diagnostic source spans and mapped models/counterexamples;
- output MIME, port-publication, staleness, and reactive-cascade results;
- worker commit and focused test commands.

The brain MUST reload persisted solves where present, rerun both paths, verify the
worker diff, and reject stale or incomplete evidence. A worker success statement
is never itself an approval signal.

### 12.4 Trust transition

The runtime profile is trusted only when all of these are true:

```text
independent_oracle
AND golden_fixtures
AND differential_match
AND brain_review_pass
```

Trust is profile-scoped by grammar version, obligation kind, lowering version,
and solver policy. Adding a new annotation, theory, verification kind, or
lowering rule returns that profile to bootstrap state until its fixtures pass.

### 12.5 Application-repair loop — the approved spec cannot move

The following native `ns-mermaid` cell is the executable application-release gate. It keeps the approved-spec/vector/adapter/repair/review flow visible and proves that release evidence is emitted only when all six immutable-spec conditions hold.

Application release requires:

```text
runtime_trusted
AND spec_approved
AND spec_hash_unchanged
AND native_proof_pass
AND conformance_pass
AND brain_review_pass
```

An implementation worker MUST NOT alter the authoritative NS cell. If the worker
believes the specification is wrong, it emits a structured change proposal and
returns the workflow to spec-authoring review. Any accepted spec edit invalidates
prior approval, conformance vectors, downstream results, and implementation
release evidence.

### 12.6 Prerequisites for dependable agent self-correction

Before this loop is an approval authority, the native runtime must provide:

1. **Cell-local port identity.** Multiple `ns-mermaid` cells cannot share global
   `spec_ir`, `proof_report`, `verified`, `conformance_vectors`, or `diagnostics`
   producer names; ports are namespaced by stable cell ID or emitted as a
   cell-local result bundle.
2. **Machine-readable diagnostics.** Parse/type/unsupported failures include
   diagnostic code, annotation ID, Mermaid node ID, exact source span, and phase.
3. **Complete proof identity.** Results include cell version, source hash, IR hash,
   obligation hash, lowering version, expected status, actual status, and source
   mapping; transient duration and `solve_id` stay outside semantic hashes.
4. **Golden differential corpus.** Positive, negative, mutation, unknown,
   timeout, cancellation, stale-output, multi-cell, port, and cascade fixtures
   compare native results with independent direct-solve expectations.
5. **Profile honesty.** The relational subset and state-machine subset advertise
   separate conformance status until `@state-var`, `@state`, `@transition`,
   `@guard`, `@update`, initiation, preservation, and bounded reachability pass
   their bootstrap fixtures.
6. **Executable reference notebook.** Formal examples in this design are
   converted from explanatory Markdown fences into real, namespaced
   `ns-mermaid` cells and run headlessly as the first conformance artifact.

In [ ]:
flowchart LR
    SPEC["`@spec NS-SPEC-IMPLEMENTATION-RELEASE-GATE
@type ReleaseDecision = enum[release, block]
@input runtime_trusted: Bool
@input spec_approved: Bool
@input spec_hash_unchanged: Bool
@input native_proof_pass: Bool
@input conformance_pass: Bool
@input brain_review_pass: Bool
@output decision: ReleaseDecision`"]

    VECTORS[Z3-generated conformance vectors]
    IMPL[real implementation through typed adapter]
    RESULT{conformance result}
    REPAIR[worker edits implementation or adapter only]
    REVIEW[brain checks runtime trust and approved hashes]

    RELEASE["`@branch RELEASE
@when runtime_trusted and spec_approved and spec_hash_unchanged and native_proof_pass and conformance_pass and brain_review_pass
@ensures RELEASE_DECISION: decision = release`"]

    BLOCK["`@branch BLOCK
@when not (runtime_trusted and spec_approved and spec_hash_unchanged and native_proof_pass and conformance_pass and brain_review_pass)
@ensures BLOCK_DECISION: decision = block`"]

    CHECK["`@verify RELEASE_DETERMINISTIC: prove determinism
@verify RELEASE_COMPLETE: prove partition_coverage
@verify RELEASE_EXCLUSIVE: prove partition_exclusive
@verify RELEASE_REACHABLE: witness branch RELEASE
@verify BLOCK_REACHABLE: witness branch BLOCK`"]

    SPEC --> VECTORS --> IMPL --> RESULT
    RESULT --> REPAIR --> IMPL
    RESULT --> REVIEW
    REVIEW --> RELEASE --> CHECK
    REVIEW --> BLOCK --> CHECK

---

## 13. Compiler-backed agent authoring of customized NS-Mermaid

**Decision.** A brain or worker MUST NOT rely on general Mermaid knowledge or a
prompt-only syntax summary when authoring NS-Mermaid. Each task binds a versioned
profile manifest, and the agent writes the sole authoritative NS-Mermaid source
inside a compiler-backed repair loop. A typed builder may produce an ephemeral
scaffold, but neither its request nor an AST may persist as a second authored
specification.

### 13.1 Approach selection and solver evidence

| Approach | Benefit | Correctness problem | Decision |
|---|---|---|---|
| Prompt plus examples | no new authoring API | permits invented annotations, invalid placement, and incomplete obligations | reject |
| Persistent AST first, Mermaid generated | syntax correct by construction | makes the AST authoritative and Mermaid derivative, creating two-source drift | reject |
| NS-Mermaid source plus compiler-backed scaffold/check | keeps the visual source authoritative while making errors machine-correctable | requires a versioned authoring facade | **adopt** |

The alternatives and acceptance gates were encoded in B′ and evaluated through
`solve_constraints` on 2026-07-30:

- `sol_cbce6a01d71d4caa`: `sat`; the accepted model selects
  `compiler_backed_ns_mermaid` and satisfies every authoring gate.
- `sol_b448c7057b4c4e22`: `unsat`; neither prompt-only raw authoring nor a
  persistent-AST-first design can be accepted under the single-source contract.
- `sol_f3f24a7bb70b4efa`: `sat`; the strict twelve-stage authoring sequence is
  feasible.

These results validate policy consistency. Parser, type-checker, formatter,
lowering, Notebook MCP, security, and end-to-end behavior still require tests.

### 13.2 Versioned authoring profile

The brain pins `profile_id`, `profile_version`, and `profile_hash` before a worker
starts. The worker reloads that exact manifest rather than reconstructing the
dialect from memory.

| Manifest field | Normative content |
|---|---|
| `mermaid_subset` | allowed diagram kinds, node/edge forms, label quoting, and prohibited constructs |
| `annotation_schema` | allowed annotations, cardinality, argument shape, and legal Mermaid attachment points |
| `expression_grammar` | types, literals, identifiers, operators, precedence, and supported theories |
| `verification_catalog` | legal `@verify` kinds, generated formula shape, and expected solver status |
| `required_obligations` | checks required for each profile feature, such as branch coverage and exclusivity |
| `canonicalization` | formatter and IR schema versions plus semantic hashing rules |
| `diagnostic_catalog` | stable codes, phases, source-span requirements, and repairability class |
| `resource_limits` | source, AST, obligation, solver-time, model, and output limits |
| `examples` | positive fixtures, one-defect negative fixtures, and their expected diagnostics/results |

A profile change creates a new hash and invalidates scaffolds, diagnostics,
proofs, and approvals produced under the previous hash.

### 13.3 Detailed authoring and correction flow

The following existing native `ns-mermaid` cell is the normative executable form of this compiler-backed authoring loop. Its rendered flow preserves the complete repair procedure, while its twelve acceptance inputs and accept/reject branches are verified in the same cell.

Only syntax, typing, placement, reference, formatting, and missing-obligation
diagnostics are automatically repairable. A solver counterexample is semantic
evidence: the worker must not weaken a declaration or delete a verification point
to make it green without an explicit brain-approved intent change.

The next cell is the normative executable acceptance gate. It is authored in the
custom NS-Mermaid dialect and makes the twelve acceptance inputs, accept/reject
branches, and five required verification points visible in one diagram.

In [ ]:
flowchart TD
    CONTEXT["`@spec AGENT-NS-MERMAID-AUTHORING-GATE
@type AuthoringDecision = enum[accept, reject]
@input profile_manifest_loaded: Bool
@input source_is_ns_mermaid: Bool
@input single_authoritative_source: Bool
@input parser_pass: Bool
@input typecheck_pass: Bool
@input canonical_roundtrip_pass: Bool
@input obligation_completeness_pass: Bool
@input solver_expectations_pass: Bool
@input diagnostics_source_mapped: Bool
@input same_cell_reexecution_match: Bool
@input semantic_hashes_match: Bool
@input brain_review_pass: Bool
@output decision: AuthoringDecision`"]

    ACCEPT["`@branch ACCEPT
@when profile_manifest_loaded and source_is_ns_mermaid and single_authoritative_source and parser_pass and typecheck_pass and canonical_roundtrip_pass and obligation_completeness_pass and solver_expectations_pass and diagnostics_source_mapped and same_cell_reexecution_match and semantic_hashes_match and brain_review_pass
@ensures DECISION_ACCEPT: decision = accept`"]

    REJECT["`@branch REJECT
@when not (profile_manifest_loaded and source_is_ns_mermaid and single_authoritative_source and parser_pass and typecheck_pass and canonical_roundtrip_pass and obligation_completeness_pass and solver_expectations_pass and diagnostics_source_mapped and same_cell_reexecution_match and semantic_hashes_match and brain_review_pass)
@ensures DECISION_REJECT: decision = reject`"]

    CHECK["`@verify AUTHORING_DECISION_DETERMINISTIC: prove determinism
@verify AUTHORING_GATE_COVERAGE: prove partition_coverage
@verify AUTHORING_GATE_EXCLUSIVE: prove partition_exclusive
@verify ACCEPT_PATH_REACHABLE: witness branch ACCEPT
@verify REJECT_PATH_REACHABLE: witness branch REJECT`"]

    CONTEXT --> ACCEPT --> CHECK
    CONTEXT --> REJECT --> CHECK

### 13.4 Executable acceptance-gate evidence

The preceding cell is a native `ns_mermaid` code cell, not an explanatory
Markdown fence. Its 2026-07-31 Notebook MCP execution emitted both
`application/vnd.spur.ns-mermaid+text` and schema-v2
`application/vnd.spur.ns-proof+json` outputs.

| Native obligation | Expected | Actual | Outcome |
|---|---:|---:|---|
| `AUTHORING_DECISION_DETERMINISTIC` | `unsat` | `unsat` | matched |
| `AUTHORING_GATE_COVERAGE` | `unsat` | `unsat` | matched |
| `AUTHORING_GATE_EXCLUSIVE` | `unsat` | `unsat` | matched |
| `ACCEPT_PATH_REACHABLE` | `sat` | `sat` | matched |
| `REJECT_PATH_REACHABLE` | `sat` | `sat` | matched |

The native report binds cell
`a33a56fe-01c9-4dc7-a5ef-be365cb2a20b` version 1 to source hash
`8ff917fe600fa204d6d1c4aa945463671c92f530a59f5605d7b16f98d7d951df`,
IR hash
`a87d9a49a11dba71958a0066e3126116172f380081817310c7dc42fe2da3c20e`,
and report hash
`a2b65a752e4a09d6b2223406855df7f8cc641c6e2b6e2ca21cf15264b68a8404`.
It records `verified=true` with five of five matched obligations. A second
Notebook MCP run preserved all semantic hashes and the status vector; transient
solver durations changed and remain outside semantic identity. Independent direct B′ encodings produced
the same status vector:

- determinism: `unsat`, `sol_3a14bd77c47e4ac2`;
- partition coverage: `unsat`, `sol_e230a021229649a3`;
- partition exclusivity: `unsat`, `sol_b3202379a0b64f5d`;
- accept witness: `sat`, `sol_2f13af4165834944`;
- reject witness: `sat`, `sol_a538d3d2520f416c`.

This is differential bootstrap evidence for the relational Boolean/enum profile.
By itself it does not extend trust to untested theories, diagnostic spans, or
future lowering versions. Section 14 adds native relational-QF_LIA,
state-machine, multi-cell, and port-namespacing confirmation while retaining the
independent-oracle requirement.

### 13.5 Normative brain/worker authoring procedure

1. **Brain defines intent.** Record semantic behavior, required branches,
   invariants, witnesses, proof expectations, and review boundaries without
   prescribing accidental syntax.
2. **Brain pins the dialect.** Attach the profile ID, version, hash, required
   obligation set, and independent `solve_id` fixtures to the worker task.
3. **Worker reloads the profile.** Failure to resolve the exact hash blocks work;
   fallback to remembered Mermaid or a nearby profile is forbidden.
4. **Worker requests a scaffold.** A profile-aware builder emits valid diagram
   topology, stable IDs, annotation placement, and typed holes. The scaffold is
   temporary; only resulting NS-Mermaid source is authoritative.
5. **Worker fills semantic holes.** It may add domain names, types, predicates,
   updates, postconditions, invariants, and named checks allowed by the manifest.
6. **Preflight parses and types.** The compiler returns canonical source,
   Canonical IR, source map, required/generated obligations, semantic hashes, and
   structured diagnostics. A failure causes a span-local repair, not a full-cell
   rewrite.
7. **Canonical round-trip runs.** `parse(format(parse(source)))` must preserve IR
   and obligation hashes. A mismatch is a compiler defect, not an agent edit cue.
8. **Obligation completeness runs.** The profile checks required proof and
   witness kinds structurally before calling the solver.
9. **Independent verification runs.** Named obligations execute through `solve`;
   persisted golden results are reloaded rather than reconstructed by the worker.
10. **Notebook MCP writes the cell.** Use `code_type: ns_mermaid`; never serialize
    an AST, generated Z3, or a second authored constraint block into the notebook.
11. **The same cell executes.** Its diagram, IR, obligations, results, diagnostics,
    and cell-local ports must match preflight semantic hashes.
12. **Brain reviews and accepts.** The brain reloads independent solves, inspects
    normalized meaning and counterexamples, checks the worker diff and profile
    hash, and records approval only when every mandatory gate is fresh.

### 13.6 Diagnostic-driven repair policy

| Evidence | Worker response | May semantics change automatically? |
|---|---|---:|
| Mermaid-subset or annotation parse error | edit only the reported span using allowed profile forms | no |
| Invalid placement, duplicate ID, unresolved reference, or type error | repair the named declaration/reference and re-run preflight | no |
| Unsupported term or verification kind | choose an equivalent supported construct or escalate a profile-change proposal | no |
| Canonical round-trip or source-map mismatch | stop; report compiler/runtime defect | no |
| Missing required obligation | add the manifest-required named `@verify` point | no |
| Proof obligation returns `sat` | attach counterexample and request semantic review | **brain approval required** |
| Witness obligation returns `unsat` | report unreachable requirement/branch for semantic review | **brain approval required** |
| `unknown`, timeout, cancellation, or internal error | simplify within approved semantics or escalate; never accept | no |
| Notebook result hash differs from preflight | treat as stale execution or runtime drift; rerun or report defect | no |
| Brain rejects normalized meaning | revise intent/source in spec-authoring state and invalidate prior evidence | **brain approval required** |

An agent MUST NOT respond to a counterexample by weakening `@requires`, deleting an
`@invariant`, widening a branch, or removing an `@verify` point merely to obtain a
green result.

### 13.7 Required authoring facade and test matrix

The standalone notebook runtime should expose a small profile-backed authoring
facade to brain and workers:

- **profile lookup:** return the exact manifest and hash;
- **scaffold:** convert structured intent and required checks into editable,
  profile-valid NS-Mermaid source;
- **preflight check:** parse, type, canonicalize, lower, enumerate obligations,
  hash, and return structured diagnostics without mutating the notebook;
- **Notebook MCP write/run:** persist only the NS-Mermaid cell source, execute the
  native cell, and return the durable output receipt;
- **evidence comparison:** compare preflight, native, and independent-oracle
  identities and statuses.

Required tests cover positive fixtures, one-defect negative fixtures, generated
scaffold validity, parse-format-parse properties, obligation completeness,
mutated lowering detection, adversarial label content, source-located repair,
unknown/timeout fail-closure, multi-cell port isolation, stale-output invalidation,
headless/UI parity, and agent attempts to delete or weaken required checks.

---

## 14. Executable complex-profile confirmation

The following native cells extend the executable reference beyond the Boolean
authoring gate. They are normative positive fixtures for the relational QF_LIA
profile and the transition-system profile. Both cells must render and publish a
schema-v2 proof report whose `cell_id`, `cell_version`, `source_hash`, `ir_hash`,
and `report_hash` match the current notebook source.

### 14.1 Relational QF_LIA transfer fixture

This cell combines Bool, Int, and enum declarations; three inputs plus an
activation flag; three outputs; three exhaustive branches; output invariants;
non-vacuity and consistency witnesses; determinism, coverage, and exclusivity
proofs; and a witness for every branch. All eight obligations are required to
match.

In [ ]:
flowchart TD
    CTX["`@spec NS-SPEC-COMPLEX-TRANSFER
@type TransferStatus = enum[success, invalid_amount, insufficient_balance]
@input source_balance: Int
@input target_balance: Int
@input amount: Int
@input account_open: Bool
@output status: TransferStatus
@output source_after: Int
@output target_after: Int
@requires PRE_SOURCE: source_balance >= 0
@requires PRE_TARGET: target_balance >= 0
@requires PRE_OPEN: account_open`"]

    INVALID["`@branch INVALID
@when amount <= 0
@ensures STATUS_INVALID: status = invalid_amount
@ensures SOURCE_INVALID: source_after = source_balance
@ensures TARGET_INVALID: target_after = target_balance`"]

    INSUFFICIENT["`@branch INSUFFICIENT
@when amount > 0 and source_balance < amount
@ensures STATUS_INSUFFICIENT: status = insufficient_balance
@ensures SOURCE_INSUFFICIENT: source_after = source_balance
@ensures TARGET_INSUFFICIENT: target_after = target_balance`"]

    SUCCESS["`@branch SUCCESS
@when amount > 0 and source_balance >= amount
@ensures STATUS_SUCCESS: status = success
@ensures SOURCE_SUCCESS: source_after = source_balance - amount
@ensures TARGET_SUCCESS: target_after = target_balance + amount`"]

    CHECK["`@invariant NONNEG_SOURCE: source_after >= 0
@invariant NONNEG_TARGET: target_after >= 0
@invariant CONSERVE: source_after + target_after = source_balance + target_balance
@verify INPUTS_NONVACUOUS: witness non_vacuity
@verify RELATION_CONSISTENT: witness consistency
@verify TRANSFER_DETERMINISTIC: prove determinism
@verify TRANSFER_COVERAGE: prove partition_coverage
@verify TRANSFER_EXCLUSIVE: prove partition_exclusive
@verify INVALID_REACHABLE: witness branch INVALID
@verify INSUFFICIENT_REACHABLE: witness branch INSUFFICIENT
@verify SUCCESS_REACHABLE: witness branch SUCCESS`"]

    CTX --> INVALID --> CHECK
    CTX --> INSUFFICIENT --> CHECK
    CTX --> SUCCESS --> CHECK

### 14.2 Transition-system fixture

This cell confirms the implemented multi-transition form: one shared declaration
and invariant note, then exactly one annotation note per transition. Explicit
`@from` and `@to` bindings make topology auditable. Initiation and preservation
are checked independently for two invariants across three transitions, yielding
eight required obligations.

In [ ]:
stateDiagram-v2
    [*] --> ACTIVE
    ACTIVE --> ACTIVE: DEBIT
    ACTIVE --> LOCKED: LOCK
    LOCKED --> ACTIVE: UNLOCK

    note right of ACTIVE
      @spec NS-SPEC-COMPLEX-ACCOUNT-STATE
      @type AccountEvent = enum[DEBIT, LOCK, UNLOCK]
      @input event: AccountEvent
      @state-var balance: Int
      @state-var locked: Bool
      @requires INIT_BALANCE: balance >= 0
      @requires INIT_UNLOCKED: not locked
      @state ACTIVE
      @state LOCKED
      @invariant BALANCE_NONNEG: balance >= 0
      @invariant LOCK_SAFE: not locked or balance >= 0
      @verify INIT_BALANCE_PROOF: prove initiate BALANCE_NONNEG
      @verify INIT_LOCK_SAFE_PROOF: prove initiate LOCK_SAFE
    end note

    note right of ACTIVE
      @transition DEBIT
      @event event = DEBIT
      @from ACTIVE
      @to ACTIVE
      @guard not locked and balance > 0
      @update balance' = balance - 1
      @update locked' = locked
      @verify DEBIT_BALANCE_PROOF: prove preserve BALANCE_NONNEG on DEBIT
      @verify DEBIT_LOCK_SAFE_PROOF: prove preserve LOCK_SAFE on DEBIT
    end note

    note right of ACTIVE
      @transition LOCK
      @event event = LOCK
      @from ACTIVE
      @to LOCKED
      @guard not locked and balance >= 0
      @update balance' = balance
      @update locked' = true
      @verify LOCK_BALANCE_PROOF: prove preserve BALANCE_NONNEG on LOCK
      @verify LOCK_LOCK_SAFE_PROOF: prove preserve LOCK_SAFE on LOCK
    end note

    note right of LOCKED
      @transition UNLOCK
      @event event = UNLOCK
      @from LOCKED
      @to ACTIVE
      @guard locked
      @update balance' = balance
      @update locked' = false
      @verify UNLOCK_BALANCE_PROOF: prove preserve BALANCE_NONNEG on UNLOCK
      @verify UNLOCK_LOCK_SAFE_PROOF: prove preserve LOCK_SAFE on UNLOCK
    end note

### 14.3 Native execution evidence and correction lessons

Notebook MCP executed all fourteen native cells in this design on 2026-08-01.
Every cell emitted `application/vnd.spur.ns-mermaid+text` plus schema-v2
`application/vnd.spur.ns-proof+json`; no obligation was inconclusive. The
thirteen positive cells matched all 111 obligations. The deliberate
`TRANSFER-ORIGINAL` negative fixture falsified its determinism claim with a
concrete Z3 counterexample, so its expected aggregate result is
`verified=false` with one mismatch.

The table retains exact proof identities for the three complex-profile fixtures:

| Native fixture | Matched | Source hash | IR hash | Report hash |
|---|---:|---|---|---|
| agent authoring gate | 5/5 | `8ff917fe600fa204d6d1c4aa945463671c92f530a59f5605d7b16f98d7d951df` | `a7df472d5e2b404b052c9315f562ca913afc62d43d33c699b2b1326d15a696d3` | `b1aa818dd3b645b3685640256eb7f689e78bdb53e7a0e6007204a8062e238018` |
| complex transfer | 8/8 | `a9528c5ca3dc608a7561007c130b23453926493df24e830fcccf203b9bf6674f` | `93bc9de40e3337aad296d3edd4e2178ab6a5139e5e122fa715e79969f91b8dd8` | `3dc54570f0563d64282d78a9ec2f5681168f2dbb33e7f67cf379bfa4e464d934` |
| complex account state | 16/16 | `44cf2d9896d919af287bcac5f5c0e5ae1f3daf31e25adb9c232ba38049bec965` | `17aaa1ac6af42e300e50f8d6b05e40cc373a380d6d73ef25783504d95f2b0ff7` | `807df0a10200a9f0336c71320410648399f20b1aefa50f5d9dac94a97426901f` |

The fourteen cells publish seventy cell-namespaced ports: one each for `spec_ir`,
`proof_report`, `verified`, `conformance_vectors`, and `diagnostics`. Their local
annotation IDs remain independent. All eleven former Markdown Mermaid fences are
now native executable cells (the authoring flow reuses its already-adjacent
native gate), so no Markdown Mermaid code fence remains in the authoritative notebook.

A separate adversarial Notebook MCP run confirmed that the verifier falsifies
incorrect declarations rather than merely accepting positive fixtures:

- a partition gap returned `sat` with `amount = 13`;
- overlap and non-determinism returned `sat` with `amount = 0` and distinct enum
  outputs;
- an unsafe decrement returned `sat` with `balance = 0` and
  `balance' = -1`;
- grouping three `@transition` declarations in one note produced source-located
  duplicate/unresolved diagnostics; splitting them into one note each repaired
  syntax without changing intent;
- proving `LOCK_SAFE` independently exposed that it could not borrow
  `BALANCE_NONNEG`; strengthening the `LOCK` guard made the invariant inductive.

These cells establish executable coverage for the implemented relational and
state-machine profiles plus the contract-driven implementation-routing policy.
They do not prove any real implementation, close the lifecycle and migration
gaps in §8.1, replace independent direct-`solve` differential fixtures, or
extend trust to unsupported theories.

---

## 15. Contract-driven implementation and conformance testing

**Decision.** SPUR uses an approved NS-Mermaid cell as an immutable executable
contract for implementation and testing. The first implementation profile is
contract-driven: Canonical IR generates a typed adapter contract and test
oracles, while a worker writes the business implementation behind that adapter.
Example-only generation is insufficient for release, and full implementation
generation from IR is deferred until its generator has an independently trusted
profile.

Z3 proves properties of the NS-Mermaid declaration and its release-routing
policy. It does not prove arbitrary implementation code. The conformance
harness is the mandatory bridge between the verified declaration and the real
implementation.

### 15.1 Approach selection

| Approach | Benefit | Correctness boundary | Decision |
|---|---|---|---|
| Generate example tests only | smallest initial surface | bounded examples can miss incorrect behavior | reject as release authority |
| Contract-driven adapter plus generated conformance | keeps the visual spec authoritative and tests real code through one typed seam | requires adapter, harness, and failure classification | **adopt** |
| Generate implementation from Canonical IR | highest automation | generator and generated code become another trusted compiler path | defer |

### 15.2 Immutable `SpecBundle` handoff

After human approval, the runner emits an immutable, content-addressed
`SpecBundle`. Every generated artifact and test report MUST bind to the same
identity tuple:

```text
(profile_id, profile_hash, cell_id, cell_version, source_hash, ir_hash,
 obligation_hashes, report_hash)
```

The bundle contains:

| Field | Purpose |
|---|---|
| profile manifest identity | pins the supported NS-Mermaid grammar, theories, lowering, and solver policy |
| Canonical IR plus source map | provides the normalized semantic contract and diagnostic locations |
| obligations and results | records expected/actual solver status and proof identity |
| conformance vectors | carries named witnesses, boundaries, and counterexamples |
| diagnostics | carries phase, code, annotation/node identity, source span, and repairability |
| adapter schema | describes typed implementation inputs, outputs, states, and events without becoming a second authored spec |

An implementation worker MUST NOT modify the approved NS-Mermaid cell or any
field of the pinned bundle. A proposed semantic change returns to spec authoring,
creates a new identity tuple, and invalidates all generated tests and release
evidence.

### 15.3 Declaration-to-test mapping

| NS-Mermaid declaration | Generated implementation evidence |
|---|---|
| `@type`, `@input`, `@output` | typed adapter interface and serialization checks |
| `@requires` | valid-input generators, rejected-input cases, and precondition assertions |
| `@branch`, `@when` | decision-table coverage and boundary partitions |
| `@ensures` | expected output assertions for each branch |
| `@invariant` | property and metamorphic tests over generated inputs |
| `@state`, `@transition` | model-based state and event traces |
| `@guard` | legal and illegal transition cases |
| `@update` | expected next-state oracle |
| witness verification points | concrete positive and boundary fixtures |
| proof counterexamples | negative regression fixtures |
| determinism, coverage, and exclusivity | mutation tests that introduce ambiguity, gaps, or overlap and MUST fail |

### 15.4 Normative implementation procedure

1. **Approve and pin.** The brain approves normalized meaning and records the
   complete proof identity tuple.
2. **Bootstrap trust.** The native parser, IR, lowering, and solver results
   match an independently maintained direct-`solve` oracle for the selected
   profile.
3. **Generate the contract.** The runner derives the adapter schema from
   Canonical IR; generated code contains no additional business semantics.
4. **Generate tests.** The runner emits witness/boundary fixtures, property
   predicates, state models, counterexample regressions, and mutation targets.
5. **Implement behind the adapter.** A worker changes implementation or adapter
   mechanics only; the pinned NS-Mermaid source remains immutable.
6. **Run conformance.** The harness invokes the real implementation through the
   typed adapter and compares outputs or next states with the generated oracle.
7. **Classify before repair.** Native/oracle disagreement routes to runtime
   repair; agreement on a proof counterexample routes to spec review; proof pass
   plus conformance failure routes to implementation or adapter repair.
8. **Release through the gate.** A release requires runtime trust, unchanged
   proof identity, a fresh native proof, complete generated artifacts, passing
   conformance/property/mutation/integration suites, and independent brain
   review.

### 15.5 Test layers and worker boundaries

| Layer | Required checks | Primary repair owner |
|---|---|---|
| Runtime bootstrap | parser/IR golden fixtures, native-versus-direct-solve differential, diagnostics, ports, staleness, reopen | NS-Mermaid runtime worker |
| Adapter contract | type mapping, serialization, invalid-input behavior, source-map identity | adapter generator worker |
| Conformance | branch witnesses, boundaries, postconditions, counterexample regressions | implementation worker |
| Properties and state | invariants, metamorphic relations, transition guards/updates, bounded traces | test-generation worker |
| Mutation | intentionally incorrect outputs, missing branches, overlaps, unsafe updates | test-generation worker |
| Integration | real service boundary, persistence, reactive cascade, proof identity, stale rejection | integration worker |
| Release review | immutable spec hash, runtime profile trust, complete reports, brain rerun | brain/reviewer |

The dependency order is runtime trust → `SpecBundle` → adapter and test
generation → implementation conformance → integration → release review. Adapter
generation and test-generation internals may proceed in parallel only after the
bundle schema and proof identity are frozen.

The following native `ns-mermaid` cell is the normative executable routing and
release gate for this procedure. Its branch witnesses demonstrate that every
failure class and the release path are reachable; determinism, coverage, and
exclusivity prove that exactly one action is selected for every gate state.

In [ ]:
flowchart TD
    SPEC["`@spec NS-SPEC-IMPLEMENTATION-CONFORMANCE-GATE
@type ImplementationDecision = enum[release, repair_runtime, reapprove_spec, repair_infrastructure, repair_implementation, await_review]
@input runtime_trusted: Bool
@input spec_identity_pinned: Bool
@input spec_hash_unchanged: Bool
@input native_proof_pass: Bool
@input adapter_contract_generated: Bool
@input vectors_generated: Bool
@input implementation_match: Bool
@input property_suite_pass: Bool
@input mutation_suite_pass: Bool
@input integration_suite_pass: Bool
@input brain_review_pass: Bool
@output decision: ImplementationDecision`"]

    VERIFY[parse lower verify and bind proof identity]
    BUNDLE[immutable SpecBundle]
    CONTRACT[generate typed adapter contract]
    TESTS[generate vectors properties state models and mutations]
    IMPL[worker implements behind adapter]
    HARNESS[run real implementation through conformance harness]
    REVIEW[brain reloads bundle and evidence]

    RUNTIME_REPAIR["`@branch RUNTIME_REPAIR
@when not runtime_trusted
@ensures ROUTE_RUNTIME: decision = repair_runtime`"]

    SPEC_REVIEW["`@branch SPEC_REVIEW
@when runtime_trusted and not (spec_identity_pinned and spec_hash_unchanged and native_proof_pass)
@ensures ROUTE_SPEC: decision = reapprove_spec`"]

    INFRA_REPAIR["`@branch INFRA_REPAIR
@when runtime_trusted and spec_identity_pinned and spec_hash_unchanged and native_proof_pass and not (adapter_contract_generated and vectors_generated)
@ensures ROUTE_INFRA: decision = repair_infrastructure`"]

    IMPLEMENTATION_REPAIR["`@branch IMPLEMENTATION_REPAIR
@when runtime_trusted and spec_identity_pinned and spec_hash_unchanged and native_proof_pass and adapter_contract_generated and vectors_generated and not (implementation_match and property_suite_pass and mutation_suite_pass and integration_suite_pass)
@ensures ROUTE_IMPLEMENTATION: decision = repair_implementation`"]

    AWAIT_REVIEW["`@branch AWAIT_REVIEW
@when runtime_trusted and spec_identity_pinned and spec_hash_unchanged and native_proof_pass and adapter_contract_generated and vectors_generated and implementation_match and property_suite_pass and mutation_suite_pass and integration_suite_pass and not brain_review_pass
@ensures ROUTE_REVIEW: decision = await_review`"]

    RELEASE["`@branch RELEASE
@when runtime_trusted and spec_identity_pinned and spec_hash_unchanged and native_proof_pass and adapter_contract_generated and vectors_generated and implementation_match and property_suite_pass and mutation_suite_pass and integration_suite_pass and brain_review_pass
@ensures ROUTE_RELEASE: decision = release`"]

    CHECK["`@verify ROUTING_DETERMINISTIC: prove determinism
@verify ROUTING_COMPLETE: prove partition_coverage
@verify ROUTING_EXCLUSIVE: prove partition_exclusive
@verify RUNTIME_REPAIR_REACHABLE: witness branch RUNTIME_REPAIR
@verify SPEC_REVIEW_REACHABLE: witness branch SPEC_REVIEW
@verify INFRA_REPAIR_REACHABLE: witness branch INFRA_REPAIR
@verify IMPLEMENTATION_REPAIR_REACHABLE: witness branch IMPLEMENTATION_REPAIR
@verify AWAIT_REVIEW_REACHABLE: witness branch AWAIT_REVIEW
@verify RELEASE_REACHABLE: witness branch RELEASE`"]

    SPEC --> VERIFY --> BUNDLE
    BUNDLE --> CONTRACT --> IMPL --> HARNESS
    BUNDLE --> TESTS --> HARNESS
    HARNESS --> REVIEW
    REVIEW --> RUNTIME_REPAIR --> CHECK
    REVIEW --> SPEC_REVIEW --> CHECK
    REVIEW --> INFRA_REPAIR --> CHECK
    REVIEW --> IMPLEMENTATION_REPAIR --> CHECK
    REVIEW --> AWAIT_REVIEW --> CHECK
    REVIEW --> RELEASE --> CHECK

### 15.6 Executable implementation-gate evidence

Notebook MCP executed the preceding native cell on 2026-08-01. It emitted
`application/vnd.spur.ns-mermaid+text` and schema-v2
`application/vnd.spur.ns-proof+json`; all nine required obligations matched
with no mismatch or inconclusive result.

- `ROUTING_DETERMINISTIC`, `ROUTING_COMPLETE`, and `ROUTING_EXCLUSIVE` were
  `unsat`, proving that the six guards select exactly one action for every
  Boolean gate state.
- Each action branch was `sat` and returned a concrete witness: runtime repair,
  spec reapproval, infrastructure repair, implementation repair, review wait,
  and release.
- Source hash:
  `ddbbcf153615076c313ce7fdc76fe9a8d9fa603e61da6ac49c30e5c126886627`.
- IR hash:
  `b5cec8fd54203313671cde79f8a27a40423003d69da122eae7b2f63af3c45cf4`.
- Report hash:
  `42c3352ee91923c0014ad771b354f225a73ff121b281f1daa895848f5fd704dc`.

This proof validates the release-routing policy, not the implementation itself.
The gate inputs MUST be populated from independently produced, proof-identity-
bound runtime, generation, conformance, property, mutation, integration, and
review reports. A worker assertion or a naked `verified=true` value is not a
valid gate input.